<a href="https://colab.research.google.com/github/LeonimerMelo/Reinforcement-Learning/blob/Policy-Gradient/Policy_Gradient_Methods_in_Reinforcement_Learning_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Policy Gradient Methods in Reinforcement Learning
Policy Gradient methods in Reinforcement Learning (RL) to directly optimize the policy, unlike value-based methods that estimate the value of states. These methods are particularly useful in environments with continuous action spaces or complex tasks where value-based approaches struggle. Given a policy $\pi$ parameterized by $\theta$, the goal is to optimize the objective:
$$ J(\theta)=\mathbb{E}\left[ \sum_{t} R_t \right] $$
Where $R_t$ is the reward at time $t$ and the expectation is taken over states and actions under the policy $\pi_{\theta}$.

###Key Advantages of Policy Gradient Methods:
- Continuous Action Spaces: Policy gradient methods can handle continuous and high-dimensional action spaces, unlike traditional value-based methods.
- Direct Optimization: These methods can directly optimize the policy without the need for approximating value functions.
- Improved Performance in Complex Environments: They perform well in environments with complex state spaces and hard-to-estimate value functions.

###Working of Policy Gradient Methods
The core idea behind policy gradient methods is to compute the gradient of the objective function $J(\theta)$ with respect to the policy parameters $\theta$. The general algorithm involves the following steps:
1. Rollout: The agent interacts with the environment following the current policy, collecting states, actions and rewards.
1. Compute the Return: The return $G_t$ is the cumulative reward obtained from time step $t$ onwards. This is often computed as the discounted sum of rewards.
1. Compute the Gradient: The gradient of the objective function with respect to the policy parameters is computed using the collected data.
1. Update the Policy: The policy parameters are updated using gradient ascent to improve the expected return.

Policy gradient helps improve decisions by checking how each action affects the total reward. Using the likelihood ratio method we adjust the policy to make better choices over time.

###Types of Policy Gradient Methods
1) **REINFORCE Algorithm**

REINFORCE is a simple Monte Carlo method that directly estimates the policy gradient using complete episodes from the environment. It updates the policy parameters based on the log probability of actions taken, weighted by the return (cumulative reward) from those actions. While simple it can suffer from high variance in the gradient estimates.

2) **Actor-Critic Methods**

Actor-Critic methods use two parts: the actor which decides what action to take and the critic which evaluates how good that action was. The critic provides feedback to the actor help to improve its decisions. This setup makes learning more stable and reduces the randomness in the updates.

3) **Proximal Policy Optimization (PPO)**

Proximal Policy Optimization (PPO) is a method that carefully updates the decision-making rules. It avoids making big changes at once which helps keep training steady. This balance makes PPO reliable and popular for tough problems.

###Challenges in Policy Gradient Methods
- High Variance: The results can change a lot, making training unstable. Actor-Critic and PPO help to fix this.
- Needs Many Samples: These methods need lots of tries to learn well.
- Local Optima: They can get stuck in not-so-great solutions and stop improving.

###Applications of Policy Gradient Methods
Policy gradient methods have shown remarkable performance in various real-world applications, including:

- Robotics: Help robots learn tasks like picking up objects, walking and moving around obstacles by learning from experience.
- Autonomous Vehicles: Policy gradient algorithms are used to optimize the driving policies for self-driving cars.
- Game AI: Enable systems to develop smart strategies by learning through trial and error in games like Go, Chess and various video games.
- Natural Language Processing: In tasks like machine translation and dialogue generation, policy gradient methods help to optimize policies for generating human-like responses.

By combining policy gradient methods with other techniques like imitation learning, exploration strategies or model-based approaches, future research could unlock even more potential in complex, real-world RL environments.

#REINFORCE Algorithm
REINFORCE is a method used in reinforcement learning to improve how decisions are made. It learns by trying actions and then adjusting the chances of those actions based on the total reward received afterward.

Unlike other methods that estimate how good each action is. REINFORCE directly learns the best way to choose actions. This makes it especially useful for tasks where there are many possible actions or continuous choices and when it is hard to estimate the value of each action.

##How REINFORCE Works
The REINFORCE algorithm works in the following steps:

- **Collect Episodes**: The agent interacts with the environment for a fixed number of steps or until an episode is complete, following the current policy. This generates a trajectory consisting of states, actions and rewards.
- **Calculate Returns**: For each time step $t$, calculate the return $G_t$ which is the total reward obtained from time $t$ onwards. Typically, this is the discounted sum of rewards:
$$G_t=\sum_{k=t}^{T}\gamma^{k-t}R_k$$
Where $\gamma$ is the discount factor, $T$ is the final time step of the episode and $R_k$ is the reward received at time step $k$.
- **Policy Gradient Update**: The policy parameters $\theta$ are updated using the following formula:
$$\theta_{t+1}=\theta_{t}+\alpha\,\nabla _{\theta}\text{log}\,\pi_{\theta}(a_t|s_t)G_t$$
Where:
 - $\alpha$ is the learning rate.
 - $\pi_{\theta}(a_t|s_t)$ is the probability of taking action $a_t$ at state $s_t$, according to the policy.
 - $G_t$ is the return or cumulative reward obtained from time step $t$ onwards.

 The gradient $\nabla _{\theta}\text{log}\,\pi_{\theta}(a_t|s_t)$ represents how much the policy probability for action  $a_t$ at state $s_t$ should be adjusted based on the obtained return.
- **Repeat**: This process is repeated for several episodes, iteratively updating the policy in the direction of higher rewards.

###Step 1: Set Up the Environment
The first step is to create the environment using OpenAI's Gym. For this example we use the CartPole-v1 environment where the agent's task is to balance a pole on a cart.

In [1]:
import gymnasium as gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

env = gym.make('CartPole-v1')
obs_space = env.observation_space.shape[0]
act_space = env.action_space.n
print('obs_space:', obs_space)
print('act_space:', act_space)

obs_space: 4
act_space: 2


In [2]:
env.spec.max_episode_steps

500

###Step 2: Define Hyperparameters
In this step we define hyperparameters for the algorithm like discount factor gamma, the learning rate, number of episodes and batch size. These hyperparameters control how the algorithm behaves during training.

In [3]:
gamma = 0.99
learning_rate = 0.01
num_episodes = 1000
batch_size = 64

###Step 3: Define the Policy Network (Actor)
We define the policy network as a simple neural network with two dense layers. The input to the network is the state and the output is a probability distribution over the actions (softmax output). The network learns the policy that maps states to action probabilities.

In [4]:
class PolicyNetwork(tf.keras.Model):
    def __init__(self, hidden_units=128):
        super(PolicyNetwork, self).__init__()
        self.dense1 = layers.Dense(hidden_units, activation='relu')
        self.dense2 = layers.Dense(act_space, activation='softmax')

    def call(self, state):
        x = self.dense1(state)
        return self.dense2(x)

###Step 4: Initialize the Policy and Optimizer
Here, we initialize the policy network and the Adam optimizer. The optimizer is used to update the weights of the policy network during training.

In [5]:
policy = PolicyNetwork()
optimizer = tf.keras.optimizers.Adam(learning_rate)

In [6]:
policy.summary()

Model: "policy_network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

###Step 5: Compute Returns
In reinforcement learning, the return $G_t$ is the discounted sum of future rewards. This function computes the return for each time step $t$, based on the rewards collected during the episode.

In [7]:
def compute_returns(rewards, gamma):
    returns = np.zeros_like(rewards, dtype=np.float32)
    running_return = 0
    for t in reversed(range(len(rewards))):
        running_return = rewards[t] + gamma * running_return
        returns[t] = running_return
    return returns

###Step 6: Define Training Step
The training step computes the gradients of the policy network using the log of action probabilities and the computed returns. The loss is the negative log-likelihood of the actions taken, weighted by the return. The optimizer updates the policy network’s parameters to maximize the expected return.

In [8]:
def train_step(states, actions, returns):
    with tf.GradientTape() as tape:
        # Calculate the probability of each action taken
        action_probs = policy(states)
        action_indices = np.array(actions, dtype=np.int32)

        # Gather the probabilities for the actions taken
        action_log_probs = tf.math.log(tf.reduce_sum(action_probs * tf.one_hot(action_indices, act_space), axis=1))

        # Calculate the loss (negative log likelihood * returns)
        loss = -tf.reduce_mean(action_log_probs * returns)

    grads = tape.gradient(loss, policy.trainable_variables)
    optimizer.apply_gradients(zip(grads, policy.trainable_variables))

###Step 7: Training Loop
The training loop collects experiences from episodes and then performs training in batches. The policy is updated after each batch of experiences. In each episode, we record the states, actions and rewards and then compute the returns. The policy is updated based on these returns.

In [9]:
for episode in range(num_episodes):
    state, _ = env.reset()
    done = False
    states, actions, rewards = [], [], []
    steps = 0
    while not done:
        state_input = np.array(state, dtype=np.float32).reshape(1, -1)
        probs = policy(state_input).numpy()[0]
        action = np.random.choice(act_space, p=probs)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        states.append(state_input[0])
        actions.append(action)
        rewards.append(reward)
        state = next_state

        steps += 1

    # After episode ends
    returns = compute_returns(rewards, gamma)
    returns = (returns - np.mean(returns)) / (np.std(returns) + 1e-9)

    states_batch = np.vstack(states)
    train_step(states_batch, actions, returns)

    if episode % 100 == 0:
        print(f"Episode {episode}/{num_episodes}, steps: {steps}")

    if steps >= env.spec.max_episode_steps:
      print('done in', episode, 'epsodes')
      break

Episode 0/1000, steps: 16
done in 62 epsodes


###Step 8: Testing the Trained Agent
After training the agent, we evaluate its performance by letting it run in the environment without updating the policy. The agent chooses actions based on the highest probabilities (greedy behavior).

In [10]:
state, _ = env.reset()
done = False
total_reward = 0

while not done:
    state_input = np.array(state, dtype=np.float32).reshape(1, -1)
    probs = policy(state_input).numpy()[0]
    action = np.argmax(probs)

    next_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    total_reward += reward
    state = next_state

print(f"Test Total Reward: {total_reward}")

Test Total Reward: 500.0


##Referências

[1] https://www.geeksforgeeks.org/machine-learning/policy-gradient-methods-in-reinforcement-learning/

[2] https://www.geeksforgeeks.org/machine-learning/reinforce-algorithm/